# Preprocess AFL event annotations

This notebook turns the verified 2023 snapshot into ball-control actions,
inferred possession episodes and a complete annotation audit. It demonstrates
rule version 0.1.0 with concrete before-and-after examples.

Episodes follow explicit project conventions rather than claiming to reproduce
the provider's possession definitions exactly. Review flags identify cases
requiring further interpretation. This stage creates neither training labels
nor a fitted model. [Preprocessing rules](../docs/preprocessing.md) describe the
feature contract and limitations.

Run the cells in order. Saved outputs provide a reference; executing the notebook
refreshes them. The original snapshot and exploration notebook are preserved.

## 1. Environment and preprocessing functions

The setup locates the repository from the current directory or its parents
and imports the preprocessing functions. The package implements the transformations
shared by the notebook and command-line interface.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

from afl_opposition.data import verify_file
from afl_opposition.preprocess import FEATURE_COLUMNS, preprocess_events, write_result

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file() and (path / "data/sources.json").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
print("Preprocessing tools loaded.")

Preprocessing tools loaded.


## 2. Raw snapshot verification

The loader verifies file size and SHA-256 against the source manifest before
reading the snapshot. Download instructions are in the README. A retained copy
supports a check that preprocessing leaves the raw DataFrame unchanged.

In [2]:
SEASON = 2023
manifest = json.loads((project_root / "data/sources.json").read_text(encoding="utf-8"))
source = manifest["seasons"][str(SEASON)]
raw_path = project_root / "data/raw" / source["filename"]
verify_file(raw_path, source)
raw = pd.read_parquet(raw_path, engine="pyarrow", dtype_backend="numpy_nullable")
raw_before = raw.copy(deep=True)
print("Snapshot:", raw_path.name)
print("Raw table:", raw.shape)

Snapshot: chains_data_2023_all.parquet
Raw table: (442482, 63)


## 3. Possession construction

The function validates event ordering, applies annotation rules and walks
forward through each match-quarter to infer possession episodes. The returned
object contains action, possession and audit tables; this cell writes no files.

Counts are operational results of the declared rules. The number of inferred
possessions can differ from the number of raw chains.

In [3]:
result = preprocess_events(raw)
events = result.events
possessions = result.possessions
annotations = result.annotations
summary_keys = [
    "raw_rows", "event_rows", "possessions", "possessions_requiring_review",
    "boundary_duplicates", "paired_kick_annotations", "unpaired_kick_annotations",
    "raw_explicit_shots", "processed_explicit_shots",
]
display(pd.Series({key: result.summary[key] for key in summary_keys}, name="count").to_frame())

,count
raw_rows,442482
event_rows,317812
possessions,56160
possessions_requiring_review,3698
boundary_duplicates,25266
paired_kick_annotations,19652
unpaired_kick_annotations,128
raw_explicit_shots,10888
processed_explicit_shots,10888


## 4. Annotation handling

The table lists every observed description, its declared role and its handling.
A description can appear more than once if some rows are retained while others
are recognised boundary copies. `review` includes deliberately unsupported
descriptions; `known_description=False` would indicate a new provider label.

Only supported ball-control and movement observations enter `events`. Fumbles,
spoils, contest targets and retrospective annotations remain in `annotations`.

In [4]:
with pd.option_context("display.max_rows", 120):
    display(result.action_audit)
print("Distinct raw descriptions:", raw["description"].nunique())
print("Unknown descriptions:", result.summary["unknown_descriptions"])

,description,role,handling,rows,known_description
0,50m Penalty For,review,review,1,True
1,Ball Up Call,stoppage,stoppage,6935,True
2,Baulked,review,review,2,True
3,Behind,stoppage,stoppage,3658,True
4,Bounce,action,event,2370,True
5,Break Operator,review,review,1,True
6,Broken Tackle,review,review,1,True
7,CB Hitout To Advantage,review,review,1,True
8,Centre Bounce,restart,restart,5896,True
9,Clearance,annotation,auxiliary_annotation,8,True


Distinct raw descriptions: 77
Unknown descriptions: []


## 5. Raw annotations and retained actions

The example uses Richmond-Carlton, quarter one, chain two. The kick at order 6 is retained;
the matching `Kick Into F50` at order 7 links to it. The kick-result and fumble
annotations remain in the audit. They are not treated as additional kicks or
as evidence that Carlton gained controlled possession.

The shot at order 12 is retained with an audit flag. That flag is not a model
feature, and future training-example construction must stop pre-shot inputs
before this event.

In [5]:
MATCH_ID = "CD_M20230140101"
QUARTER = 1
CHAIN_NUMBER = 2
raw_selection = (
    annotations["match_id"].eq(MATCH_ID)
    & annotations["period"].eq(QUARTER)
    & annotations["chain_number"].eq(CHAIN_NUMBER)
)
display(annotations.loc[raw_selection, [
    "display_order", "period_seconds", "description", "team_id", "handling"
]])
event_selection = (
    events["match_id"].eq(MATCH_ID)
    & events["period"].eq(QUARTER)
    & events["chain_number"].eq(CHAIN_NUMBER)
)
display(events.loc[event_selection, [
    "display_order", "description", "action_type", "team_id",
    "x_attacking", "y_attacking", "gap_seconds", "is_explicit_shot",
]])

,display_order,period_seconds,description,team_id,handling
4,5,24,Free For,CD_T120,event
5,6,26,Kick,CD_T120,event
6,7,26,Kick Into F50,CD_T120,paired_kick_annotation
7,8,27,Kick Inside 50 Result,<NA>,auxiliary_annotation
8,9,29,Mark Fumbled,CD_T30,context_only
9,10,29,Mark Dropped,CD_T30,context_only
10,11,32,Loose Ball Get Crumb,CD_T120,event
11,12,32,Kick,CD_T120,event
12,13,35,Goal,CD_T120,stoppage


,display_order,description,action_type,team_id,x_attacking,y_attacking,gap_seconds,is_explicit_shot
2,5,Free For,free_kick,CD_T120,-3,-2,0,False
3,6,Kick,kick,CD_T120,-8,-7,2,False
4,11,Loose Ball Get Crumb,loose_ball_get,CD_T120,46,-5,6,False
5,12,Kick,kick,CD_T120,52,-5,0,True


## 6. Duplicate boundary annotations

Orders 34 and 35 describe the same Richmond possession gain in adjacent raw
chains. Their coordinates have opposite signs because the chain coordinate
frame changes from Carlton to Richmond. Matching identity, timestamp and
frame-consistent location support keeping the first observation only.

The retained event is oriented towards Richmond's attacking direction. No
future observation is used to move its position or change its features.

In [6]:
boundary_selection = (
    annotations["match_id"].eq(MATCH_ID)
    & annotations["period"].eq(QUARTER)
    & annotations["display_order"].between(34, 36)
)
display(annotations.loc[boundary_selection, [
    "display_order", "chain_number", "description", "team_id",
    "chain_team_id", "x", "y", "handling", "canonical_event_id",
]])
display(events.loc[
    events["match_id"].eq(MATCH_ID) & events["period"].eq(QUARTER)
    & events["display_order"].between(34, 36),
    ["display_order", "raw_x", "raw_y", "x_attacking", "y_attacking", "x_norm", "y_norm"],
])

,display_order,chain_number,description,team_id,chain_team_id,x,y,handling,canonical_event_id
33,34,4,Loose Ball Get Crumb,CD_T120,CD_T30,39,-8,event,CD_M20230140101:q1:c4:e34
34,35,5,Loose Ball Get Crumb,CD_T120,CD_T120,-39,8,boundary_duplicate,CD_M20230140101:q1:c4:e34
35,36,5,Kick,CD_T120,CD_T120,-37,10,event,CD_M20230140101:q1:c5:e36


,display_order,raw_x,raw_y,x_attacking,y_attacking,x_norm,y_norm
18,34,39,-8,-39,8,-0.4875,0.113475
19,36,-37,10,-37,10,-0.4625,0.141844


## 7. Inferred possession structure

This example uses the possession containing the first retained action from
the selected chain. Episode identifiers refer to their first observed action and can span
raw-chain boundaries. The episode ends on supported opponent control, a
stoppage or a flagged unresolved boundary.

`event_index` is zero-based. Time gaps refer to retained actions, not every raw
annotation. Both gap and elapsed time are zero at the episode's first action.

In [7]:
POSSESSION_ID = events.loc[event_selection, "possession_id"].iloc[0]
display(possessions.loc[possessions["possession_id"].eq(POSSESSION_ID)].T)
display(events.loc[events["possession_id"].eq(POSSESSION_ID), [
    "event_index", "display_order", "description", "period_seconds",
    "gap_seconds", "elapsed_seconds", "is_explicit_shot",
]])

,1
possession_id,CD_M20230140101:q1:c2:e5:pos
match_id,CD_M20230140101
period,1
team_id,CD_T120
start_event_id,CD_M20230140101:q1:c2:e5
last_event_id,CD_M20230140101:q1:c2:e12
start_seconds,24
last_event_seconds,32
closed_at_seconds,35
end_reason,Goal


,event_index,display_order,description,period_seconds,gap_seconds,elapsed_seconds,is_explicit_shot
2,0,5,Free For,24,0,0,False
3,1,6,Kick,26,2,2,False
4,2,11,Loose Ball Get Crumb,32,6,8,False
5,3,12,Kick,32,0,8,True


## 8. Review flags and exclusions

Review flags expose unsupported annotations, incomplete team information,
unresolved boundaries, unpaired kick annotations and coordinate anomalies.
End-of-input episodes are censored rather than assumed to end without a shot.
An episode can have multiple reasons, so these counts need not sum to the
number of flagged episodes.

Review metadata can depend on later rows. It must not be fed to a model.
Excluding flagged episodes also changes the sample and requires coverage
reporting before evaluation.

In [8]:
review = possessions.loc[possessions["requires_review"]]
review_counts = review["review_reasons"].str.split(";").explode().value_counts()
display(review_counts.rename("episodes").to_frame())
print("Episodes requiring review:", f"{len(review) / len(possessions):.1%}")
display(review[[
    "match_id", "period", "team_id", "event_count", "end_reason", "review_reasons"
]].head(10))

,episodes
review_reasons,
unsupported_description,2075
unresolved_chain_change,884
end_of_available_events,622
unpaired_kick_annotation,202
missing_action_team,4
coordinate_jump_over_100m,1


Episodes requiring review: 6.6%


,match_id,period,team_id,event_count,end_reason,review_reasons
7,CD_M20230140101,1,CD_T30,6,opponent_control,missing_action_team
8,CD_M20230140101,1,CD_T120,4,opponent_control,missing_action_team
11,CD_M20230140101,1,CD_T120,6,unresolved_chain_change,unresolved_chain_change
16,CD_M20230140101,1,CD_T30,15,Goal,unsupported_description
28,CD_M20230140101,1,CD_T120,6,unresolved_chain_change,unresolved_chain_change
43,CD_M20230140101,1,CD_T30,2,unresolved_chain_change,unresolved_chain_change
49,CD_M20230140101,1,CD_T30,1,unresolved_chain_change,unresolved_chain_change
52,CD_M20230140101,1,CD_T120,4,unresolved_chain_change,unresolved_chain_change
72,CD_M20230140101,1,CD_T120,9,end_of_available_events,end_of_available_events
79,CD_M20230140101,2,CD_T120,3,opponent_control,unsupported_description


## 9. Feature contract and structural checks

The initial feature allowlist contains the action category, normalised
coordinates and past-only timing. Event order supplies positional information
to a sequence model. Shot flags, end reasons, review flags and episode totals
are audit or target-construction metadata.

The checks below verify raw-table preservation, identifiers, single-team
episodes and non-negative elapsed times. Shot-flag preservation is checked on
the recorded 2023 snapshot; it does not establish the completeness of shot
annotation. No targets are assigned from missing flags.

In [9]:
display(events.loc[events["possession_id"].eq(POSSESSION_ID), FEATURE_COLUMNS])
pd.testing.assert_frame_equal(raw, raw_before)
assert len(annotations) == len(raw)
assert events["event_id"].is_unique
assert possessions["possession_id"].is_unique
assert events.groupby("possession_id")["team_id"].nunique().eq(1).all()
assert events.groupby("possession_id")["match_id"].nunique().eq(1).all()
assert events.groupby("possession_id")["period"].nunique().eq(1).all()
assert events["gap_seconds"].ge(0).all()
assert events["elapsed_seconds"].ge(0).all()
pd.testing.assert_series_equal(
    events.groupby("possession_id").size().sort_index(),
    possessions.set_index("possession_id")["event_count"].sort_index(),
    check_dtype=False, check_names=False,
)
assert result.summary["raw_explicit_shots"] == result.summary["processed_explicit_shots"]
print("Structural checks passed; raw DataFrame unchanged.")

,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds
2,free_kick,-0.0375,-0.028369,0,0
3,kick,-0.1,-0.099291,2,2
4,loose_ball_get,0.575,-0.070922,6,8
5,kick,0.65,-0.070922,0,8


Structural checks passed; raw DataFrame unchanged.


## 10. Derived tables and provenance

The export writes four Parquet tables and a JSON summary beneath
`data/processed/2023/`.
`overwrite=True` regenerates these derived files on subsequent notebook runs.
The summary records the source checksum and rule version. The raw snapshot
is verified again after writing; only relative output names are displayed.

These files support the next stage: reviewed label construction and
chronological dataset splits. They are not yet training-ready tensors.

In [10]:
output_dir = project_root / "data/processed" / str(SEASON)
write_result(result, output_dir, raw_path, overwrite=True)
verify_file(raw_path, source)
for filename in ["events.parquet", "possessions.parquet", "annotations.parquet", "action_audit.parquet", "summary.json"]:
    print((output_dir / filename).relative_to(project_root).as_posix())
print("Raw snapshot checksum unchanged.")

data/processed/2023/events.parquet
data/processed/2023/possessions.parquet
data/processed/2023/annotations.parquet
data/processed/2023/action_audit.parquet
data/processed/2023/summary.json
Raw snapshot checksum unchanged.
